# Prompt 10 — Custom Conditions and Sensitive Data
## Terraform Associate (004) Exam Study Guide

**Exam Objective:** Write and interpret variable validation, custom lifecycle conditions, check blocks, and sensitive data patterns in Terraform HCL.

---

## Table of Contents
1. [Variable `validation` Block](#1-variable-validation-block)
2. [`check` Block (Terraform 1.5+)](#2-check-block-terraform-15)
3. [`precondition` and `postcondition` Blocks](#3-precondition-and-postcondition-blocks)
4. [Comparing All Three Condition Mechanisms](#4-comparing-all-three-condition-mechanisms)
5. [Sensitive Data Management](#5-sensitive-data-management)
6. [HashiCorp Vault Integration](#6-hashicorp-vault-integration)
7. [Exam-Style Questions](#7-exam-style-questions)
8. [Real-World Scenarios](#8-real-world-scenarios)
9. [Quick-Reference Cheat Sheet](#9-quick-reference-cheat-sheet)

---
## 1. Variable `validation` Block

The `validation` block lets you enforce constraints on input variable values **before Terraform evaluates the rest of the configuration**. Validation errors are reported immediately at plan time.

### Syntax

```hcl
variable "environment" {
  type        = string
  description = "Deployment environment name"

  validation {
    condition     = <boolean expression>
    error_message = "Human-readable message shown when condition is false."
  }
}
```

### Rules

| Rule | Detail |
|---|---|
| `condition` type | Must evaluate to `bool` — if `false`, Terraform errors |
| `error_message` | Must be a non-empty string; shown to the user when validation fails |
| Variable reference | May only reference `var.<name>` — **cannot** reference other variables or resources |
| Multiple blocks | A variable may have **more than one** `validation` block — each is checked independently |
| Timing | Checked before plan/apply begins; catches bad input early |

### Using `can()` in Conditions

`can()` returns `true` if an expression evaluates without error — it wraps type-conversion attempts so the condition doesn't itself error when the input is the wrong type.

```hcl
# Without can(): if var.port is not a number-like string, the tonumber() call
# would raise an error INSIDE the condition, which is different from
# the condition returning false.
condition = tonumber(var.port) >= 1 && tonumber(var.port) <= 65535

# With can(): gracefully returns false if tonumber() fails
condition = can(tonumber(var.port)) && tonumber(var.port) >= 1 && tonumber(var.port) <= 65535
```

In [ ]:
# ── VARIABLE VALIDATION EXAMPLES ─────────────────────────────────────────────

validation_examples_hcl = '''
# ── Example 1: Allowlist of string values ────────────────────────────────────
variable "environment" {
  type        = string
  description = "Deployment environment"

  validation {
    condition     = contains(["dev", "staging", "prod"], var.environment)
    error_message = "environment must be one of: dev, staging, prod."
  }
}

# ── Example 2: String length constraint ──────────────────────────────────────
variable "bucket_name" {
  type = string

  validation {
    condition     = length(var.bucket_name) >= 3 && length(var.bucket_name) <= 63
    error_message = "S3 bucket names must be between 3 and 63 characters."
  }

  validation {
    # Bucket names must start with a lowercase letter or digit
    condition     = can(regex("^[a-z0-9]", var.bucket_name))
    error_message = "Bucket name must start with a lowercase letter or digit."
  }
}

# ── Example 3: Numeric range using can() ─────────────────────────────────────
variable "replica_count" {
  type = number

  validation {
    condition     = var.replica_count >= 1 && var.replica_count <= 20
    error_message = "replica_count must be between 1 and 20."
  }
}

# ── Example 4: CIDR block format validation ───────────────────────────────────
variable "vpc_cidr" {
  type = string

  validation {
    # can() gracefully handles the case where cidrhost() would error
    # on a string that is not a valid CIDR block.
    condition     = can(cidrhost(var.vpc_cidr, 0))
    error_message = "vpc_cidr must be a valid IPv4 CIDR block (e.g. 10.0.0.0/16)."
  }
}

# ── Example 5: Object variable with validated fields ──────────────────────────
variable "tags" {
  type = object({
    environment = string
    owner       = string
  })

  validation {
    condition     = length(var.tags.owner) > 0
    error_message = "tags.owner must not be empty — every resource must have an owner."
  }
}
'''

print("Key patterns:")
print("  contains([allowed], var.x)   — allowlist check")
print("  length(var.x) >= N           — length constraint")
print("  can(regex(..., var.x))       — format check, safe with can()")
print("  can(cidrhost(var.x, 0))      — CIDR validation")
print()
print(validation_examples_hcl)

---
## 2. `check` Block (Terraform 1.5+)

The `check` block asserts conditions about infrastructure **after** apply completes. Unlike `precondition`/`postcondition`, a failing `check` produces a **warning** — it does not fail the plan or apply.

### When to Use
- Verify that a web server is actually responding after provisioning
- Assert that an S3 bucket policy has the expected settings
- Continuous health checks that run on every plan/apply without blocking

### Syntax

```hcl
check "<label>" {
  data "<type>" "<name>" {
    # Optional: a data source scoped to this check block only
    # This data source is NOT accessible elsewhere in the config
  }

  assert {
    condition     = <boolean expression>
    error_message = "Message shown when assertion fails."
  }
}
```

### Key Facts

| Fact | Detail |
|---|---|
| Severity | **Warning only** — apply succeeds even if assertion fails |
| Timing | Evaluated after all resources are applied |
| Scoped data source | The `data` block inside `check` is evaluated for assertion only — **not added to state** |
| Multiple asserts | A single `check` block may contain multiple `assert` blocks |
| Available since | Terraform 1.5 |

> **Exam tip:** `check` block = **soft assertion / warning**. If you see a question asking which mechanism produces a warning (not an error), the answer is `check`.

In [ ]:
# ── CHECK BLOCK EXAMPLES ──────────────────────────────────────────────────────

check_block_hcl = '''
# ── Example 1: HTTP health check after creating a web server ─────────────────
resource "aws_instance" "web" {
  ami           = var.ami_id
  instance_type = "t3.micro"
  user_data     = file("${path.module}/startup.sh")
}

resource "aws_eip" "web" {
  instance = aws_instance.web.id
}

# The check block runs after the above resources are applied.
# It reads the HTTP endpoint and asserts a 200 response.
# A non-200 response produces a WARNING but does not fail the apply.
check "web_health" {
  data "http" "endpoint" {
    url = "http://${aws_eip.web.public_ip}/health"
  }

  assert {
    condition     = data.http.endpoint.status_code == 200
    error_message = "Web server health check failed: expected HTTP 200, got ${data.http.endpoint.status_code}."
  }
}

# ── Example 2: Assert that an S3 bucket has versioning enabled ───────────────
resource "aws_s3_bucket" "data" {
  bucket = "my-critical-data"
}

resource "aws_s3_bucket_versioning" "data" {
  bucket = aws_s3_bucket.data.id
  versioning_configuration {
    status = "Enabled"
  }
}

check "s3_versioning_enabled" {
  data "aws_s3_bucket" "data_check" {
    bucket = aws_s3_bucket.data.bucket
  }

  assert {
    condition     = data.aws_s3_bucket.data_check.versioning_enabled == true
    error_message = "S3 bucket ${aws_s3_bucket.data.bucket} does not have versioning enabled."
  }
}
'''

print("check block behaviour summary:")
print("  check assertion PASSES  → no output")
print("  check assertion FAILS   → Warning message printed; apply still succeeds")
print("  precondition FAILS      → Hard error; plan/apply aborted")
print()
print(check_block_hcl)

---
## 3. `precondition` and `postcondition` Blocks

`precondition` and `postcondition` are nested inside a resource's `lifecycle` block. They produce **hard errors** — failing either will abort the plan or apply.

### `precondition`
- Checked **before** the resource is created or updated
- Validates assumptions the resource depends on — e.g., the AMI ID belongs to the expected OS family
- If it fails: Terraform errors and does **not** proceed with creating/updating the resource

### `postcondition`
- Checked **after** the resource is created or updated
- Validates invariants — e.g., the created load balancer has a DNS name
- If it fails: Terraform errors, rolls back what it can, and reports the violation

### Syntax

```hcl
resource "<type>" "<name>" {
  # ...

  lifecycle {
    precondition {
      condition     = <boolean expression>
      error_message = "Message if precondition fails."
    }

    postcondition {
      condition     = <boolean expression>
      error_message = "Message if postcondition fails."
    }
  }
}
```

### Expression Scope

| Block | Can Reference |
|---|---|
| `precondition` | `var.*`, `data.*`, `local.*`, other resource attributes |
| `postcondition` | Same as above, **plus** `self.*` — the resource's own attributes after creation |

> **Exam tip:** `postcondition` can use `self` to reference the resource's own attributes. `precondition` cannot — the resource doesn't exist yet.

In [ ]:
# ── PRECONDITION AND POSTCONDITION EXAMPLES ───────────────────────────────────

conditions_hcl = '''
# ── Example 1: precondition — verify AMI architecture ────────────────────────
data "aws_ami" "app" {
  most_recent = true
  owners      = ["amazon"]

  filter {
    name   = "name"
    values = ["amzn2-ami-hvm-*-x86_64-gp2"]
  }
}

resource "aws_instance" "app" {
  ami           = data.aws_ami.app.id
  instance_type = var.instance_type

  lifecycle {
    # Checked before the instance is created.
    # Ensures the AMI is x86_64 — catches mismatches before wasting time on creation.
    precondition {
      condition     = data.aws_ami.app.architecture == "x86_64"
      error_message = "AMI ${data.aws_ami.app.id} is not x86_64. Choose a compatible instance type or AMI."
    }
  }
}

# ── Example 2: postcondition — verify the ALB got a DNS name ─────────────────
resource "aws_lb" "api" {
  name               = "api-alb"
  load_balancer_type = "application"
  subnets            = var.public_subnet_ids

  lifecycle {
    # Checked after the load balancer is created.
    # self.dns_name refers to the attribute of this resource after creation.
    postcondition {
      condition     = self.dns_name != ""
      error_message = "Load balancer was created but has no DNS name — check VPC and subnet configuration."
    }
  }
}

# ── Example 3: Both in a module — documenting invariants ─────────────────────
resource "aws_db_instance" "primary" {
  identifier        = var.db_identifier
  engine            = "postgres"
  instance_class    = var.db_instance_class
  allocated_storage = var.db_storage_gb
  username          = var.db_username
  password          = var.db_password

  lifecycle {
    # Precondition: production must use at least db.t3.small
    precondition {
      condition     = var.environment != "prod" || contains(["db.t3.small", "db.m5.large", "db.m5.xlarge"], var.db_instance_class)
      error_message = "Production RDS must use db.t3.small or larger."
    }

    # Postcondition: verify the endpoint was assigned
    postcondition {
      condition     = self.endpoint != ""
      error_message = "RDS instance created without an endpoint — investigate VPC/subnet configuration."
    }
  }
}
'''

print("Timing and severity:")
print("  precondition  → BEFORE resource create/update → HARD ERROR if fails")
print("  postcondition → AFTER  resource create/update → HARD ERROR if fails")
print("  check assert  → AFTER  all resources applied  → WARNING  if fails")
print()
print(conditions_hcl)

---
## 4. Comparing All Three Condition Mechanisms

| Feature | `variable validation` | `precondition` | `postcondition` | `check` |
|---|---|---|---|---|
| **Where defined** | Inside `variable` block | Inside `lifecycle` block | Inside `lifecycle` block | Top-level `check` block |
| **When evaluated** | Before plan begins | Before resource create/update | After resource create/update | After all resources applied |
| **On failure** | Hard error | Hard error | Hard error | Warning only |
| **Can reference `self`** | No | No | Yes | No |
| **Can contain data source** | No | No | No | Yes (scoped) |
| **Use case** | Validate user input | Guard resource assumptions | Verify resource invariants | Soft health checks |
| **Available since** | Terraform 0.13 | Terraform 1.2 | Terraform 1.2 | Terraform 1.5 |

### Decision Tree

```
Is this about validating user INPUT (a variable value)?
  YES → variable validation block

Is this about an ASSUMPTION that must be true BEFORE creating a resource?
  YES → precondition

Is this about an INVARIANT that must hold AFTER a resource is created?
  YES → postcondition

Is this a SOFT HEALTH CHECK that should warn but not block?
  YES → check block
```

---
## 5. Sensitive Data Management

Terraform has multiple mechanisms to prevent sensitive values from being accidentally exposed in logs, terminal output, or plan files.

### 5.1 `sensitive = true` on Variables

```hcl
variable "db_password" {
  type        = string
  description = "Database master password"
  sensitive   = true   # value is masked in plan/apply output
}
```

**Effect:**
- Value is replaced with `(sensitive value)` in `terraform plan` and `terraform apply` output
- Value is replaced with `(sensitive)` in `terraform output`
- Value is still stored **in plaintext** in the state file

### 5.2 `sensitive = true` on Outputs

```hcl
output "db_connection_string" {
  value     = "postgres://${var.db_user}:${var.db_password}@${aws_db_instance.prod.endpoint}/app"
  sensitive = true
}
```

**Effect:**
- Masked in `terraform apply` output: `db_connection_string = (sensitive value)`
- `terraform output db_connection_string` is also masked
- `terraform output -json` and `terraform output -raw` **do** reveal the value — useful for automation
- Still in state file in plaintext

### 5.3 Sensitive Value Propagation

Terraform **automatically marks a value as sensitive** if it is derived from a sensitive value — even if the derived output doesn't explicitly set `sensitive = true`.

```hcl
# var.db_password is sensitive
# This local is automatically treated as sensitive:
locals {
  connection_url = "postgres://${var.db_password}@db.example.com/app"
}
```

> **Exam warning:** An output that uses a sensitive variable will cause Terraform to error unless the output also has `sensitive = true` declared.

### 5.4 `sensitive()` Function

Explicitly marks any value as sensitive:

```hcl
locals {
  # Mark a value sensitive even though the source variable is not
  api_key = sensitive("hardcoded-value")   # avoid this pattern — use Vault instead
}
```

### 5.5 `nonsensitive()` Function

Explicitly removes the sensitive marking — use with extreme caution:

```hcl
output "db_username" {
  # var.db_username was marked sensitive, but the username is not secret
  # Use nonsensitive() to allow it to appear in output
  value = nonsensitive(var.db_username)
}
```

> **Exam warning:** `nonsensitive()` should only be used when you are **certain** the value is not secret. Misuse can accidentally expose credentials.

### 5.6 State File Security — Critical Exam Topic

| Fact | Detail |
|---|---|
| Sensitive values in state | Stored in **plaintext** in `terraform.tfstate` — regardless of `sensitive = true` |
| `sensitive = true` protection | Masks terminal/plan output only — does NOT encrypt state |
| Mitigation | Use encrypted remote backends (S3 with SSE, HCP Terraform) |
| Never do | Commit `terraform.tfstate` to version control |

In [ ]:
# ── SENSITIVE DATA EXAMPLE ────────────────────────────────────────────────────

sensitive_hcl = '''
# ── variables.tf ─────────────────────────────────────────────────────────────
variable "db_username" {
  type      = string
  sensitive = true
}

variable "db_password" {
  type      = string
  sensitive = true
}

# ── main.tf ──────────────────────────────────────────────────────────────────
resource "aws_db_instance" "prod" {
  identifier        = "prod-db"
  engine            = "postgres"
  instance_class    = "db.t3.medium"
  allocated_storage = 100
  username          = var.db_username
  password          = var.db_password
}

# ── outputs.tf ───────────────────────────────────────────────────────────────
output "db_endpoint" {
  description = "RDS endpoint for application configuration"
  value       = aws_db_instance.prod.endpoint
  # endpoint is not derived from a sensitive variable, so no sensitive = true needed
}

output "db_connection_string" {
  description = "Full connection string (sensitive — contains credentials)"
  # This output derives from sensitive variables, so Terraform REQUIRES
  # sensitive = true or it will error.
  value     = "postgres://${var.db_username}:${var.db_password}@${aws_db_instance.prod.endpoint}/app"
  sensitive = true
}

# Accessing sensitive output in automation:
#   terraform output -json db_connection_string   → reveals value in JSON
#   terraform output -raw  db_connection_string   → reveals raw string
#   terraform output       db_connection_string   → shows "(sensitive value)"
'''

print("Sensitive behaviour summary:")
print("  sensitive = true   → masks output in terminal and plan")
print("  state file         → ALWAYS stores plaintext — encrypt your backend!")
print("  nonsensitive()     → removes mask — use with extreme caution")
print("  sensitive()        → adds mask to any value")
print()
print(sensitive_hcl)

---
## 6. HashiCorp Vault Integration

HashiCorp Vault is a secrets management platform. Integrating Terraform with Vault ensures that secrets are **never stored in `.tfvars` files, environment variables, or version control**.

### The Vault Provider

The `vault` Terraform provider reads secrets from Vault as **data sources** at plan/apply time.

```hcl
terraform {
  required_providers {
    vault = {
      source  = "hashicorp/vault"
      version = "~> 3.0"
    }
  }
}

provider "vault" {
  address = "https://vault.example.com"
  # Authentication via VAULT_TOKEN environment variable or other auth methods
}

# Read a KV secret from Vault
data "vault_kv_secret_v2" "db_creds" {
  mount = "kv"
  name  = "prod/database"
}

resource "aws_db_instance" "prod" {
  username = data.vault_kv_secret_v2.db_creds.data["username"]
  password = data.vault_kv_secret_v2.db_creds.data["password"]
  # Other configuration...
}
```

### Benefits

| Benefit | Detail |
|---|---|
| No secrets in VCS | Secrets fetched at runtime — never committed |
| Centralised rotation | Rotate secrets in Vault; Terraform reads fresh values on next apply |
| Audit trail | Vault logs every secret access with who, what, and when |
| Dynamic credentials | Vault can generate short-lived AWS credentials instead of static IAM keys |

### HCP Terraform Dynamic Credentials

HCP Terraform (formerly Terraform Cloud) supports **dynamic provider credentials** via Vault and OIDC:

1. HCP Terraform workspace is configured with a Vault-backed OIDC trust relationship
2. Each run gets **short-lived, automatically-rotated** cloud provider credentials
3. No static AWS access keys or GCP service account keys are stored in workspace variables

> **Exam tip:** Dynamic credentials = Vault generates short-lived credentials per run via OIDC. Static credentials = long-lived API keys stored in workspace variables (less secure).

In [ ]:
# ── VAULT INTEGRATION EXAMPLE ─────────────────────────────────────────────────

vault_hcl = '''
# ── Vault provider + data source ─────────────────────────────────────────────
provider "vault" {
  address = var.vault_address
  # VAULT_TOKEN env var authenticates by default
  # Can also use token auth, AppRole, AWS auth method, etc.
}

# Read database credentials from Vault KV v2 engine
data "vault_kv_secret_v2" "rds_creds" {
  mount = "secret"
  name  = "${var.environment}/rds/primary"
}

resource "aws_db_instance" "primary" {
  identifier        = "${var.environment}-postgres"
  engine            = "postgres"
  instance_class    = "db.t3.medium"
  allocated_storage = 50

  # Credentials fetched from Vault — never in .tfvars or version control
  username = data.vault_kv_secret_v2.rds_creds.data["username"]
  password = data.vault_kv_secret_v2.rds_creds.data["password"]

  skip_final_snapshot = var.environment != "prod"
}

# ── Dynamic AWS credentials from Vault ───────────────────────────────────────
# Vault AWS Secrets Engine generates short-lived IAM credentials per apply
data "vault_aws_access_credentials" "deploy" {
  backend = "aws"
  role    = "terraform-deploy-role"
}

provider "aws" {
  access_key = data.vault_aws_access_credentials.deploy.access_key
  secret_key = data.vault_aws_access_credentials.deploy.secret_key
  token      = data.vault_aws_access_credentials.deploy.security_token
  region     = var.aws_region
}
'''

print("Vault integration patterns:")
print("  vault_kv_secret_v2    → static secrets from KV store")
print("  vault_aws_access_creds → dynamic, short-lived AWS credentials")
print("  Both avoid storing secrets in .tfvars or environment variables")
print()
print(vault_hcl)

---
## 7. Exam-Style Questions

Test yourself — click **Answer** to reveal each explanation.

### Question 1

A Terraform configuration defines the following:

```hcl
variable "db_password" {
  type      = string
  sensitive = true
}

output "connection_string" {
  value = "postgres://${var.db_password}@db.example.com/app"
}
```

What happens when `terraform plan` is run?

**A.** The plan succeeds and the password is masked in the output value.

**B.** Terraform errors because an output references a sensitive variable without `sensitive = true` on the output.

**C.** The plan succeeds and the password appears in plaintext in the plan output.

**D.** Terraform automatically adds `sensitive = true` to the output and the plan succeeds.

<details><summary>Answer</summary>

**Answer: B**

When a Terraform output references a sensitive value, Terraform requires the output to be explicitly marked `sensitive = true`. Without it, Terraform errors:

```
Error: Output refers to sensitive values
  on outputs.tf line 4:
  │ output "connection_string" {
  │
  │ To avoid accidentally exporting sensitive data that was intended to be
  │ only internal, Terraform requires that any root module output containing
  │ sensitive data be explicitly marked as sensitive, to confirm your intent.
```

The fix is:
```hcl
output "connection_string" {
  value     = "postgres://${var.db_password}@db.example.com/app"
  sensitive = true
}
```

- A is incorrect: Terraform does not auto-mask — it errors instead.
- C is incorrect: Terraform never exposes sensitive values in plaintext output.
- D is incorrect: Terraform does not automatically add `sensitive = true` — it requires explicit declaration.

</details>

### Question 2

A team adds a `check` block to verify that a newly created web server responds with HTTP 200. After running `terraform apply`, the web server is still starting up and the HTTP check returns 503. What is the result?

**A.** The apply fails and the web server resource is rolled back.

**B.** The apply succeeds and a warning is shown about the failed assertion.

**C.** The apply fails and Terraform exits with a non-zero code.

**D.** The check block is skipped because the resource is not yet ready.

<details><summary>Answer</summary>

**Answer: B**

The `check` block produces a **warning** when its assertion fails — the apply still succeeds. This is the key behavioural difference between `check` (soft assertion) and `precondition`/`postcondition` (hard errors).

The output will look something like:

```
╷
│ Warning: Check block assertion failed
│
│   on main.tf line 20, in check "web_health":
│    20:     condition = data.http.endpoint.status_code == 200
│
│ Web server health check failed: expected HTTP 200, got 503.
╵

Apply complete! Resources: 1 added, 0 changed, 0 destroyed.
```

- A is incorrect: `check` block failures never roll back resources.
- C is incorrect: the apply exits with code 0 (success) even with a failed check assertion.
- D is incorrect: `check` blocks are always evaluated after apply, regardless of resource state.

</details>

### Question 3

A module uses a `postcondition` block on an RDS instance to verify the endpoint is not empty. The RDS creation succeeds in AWS, but the `postcondition` fails because the endpoint attribute is unexpectedly empty. What happens?

**A.** Terraform issues a warning and continues with the rest of the apply.

**B.** Terraform errors and the RDS instance remains in state as a created resource.

**C.** Terraform errors and rolls back (destroys) the RDS instance automatically.

**D.** Terraform marks the resource as tainted so it will be recreated on the next apply.

<details><summary>Answer</summary>

**Answer: B**

A failing `postcondition` is a **hard error** — Terraform aborts the apply and reports the error. However, the resource **was successfully created in AWS** and is recorded in state. Terraform does not automatically destroy it.

The error message will reference the `postcondition` condition and the `error_message` string. The operator must investigate why the endpoint is empty (likely a VPC/subnet configuration issue) and then re-run apply or manually remediate.

Key contrast:
- `check` block failure → warning, apply succeeds
- `postcondition` failure → hard error, apply fails, resource left in state as-is

- A is incorrect: `postcondition` is a hard error, not a warning (that is the behaviour of `check`).
- C is incorrect: Terraform does not automatically destroy resources when a postcondition fails.
- D is incorrect: Terraform does not automatically taint resources on postcondition failure.

</details>

---
## 8. Real-World Scenarios

<details>
<summary>Scenario 1: Enforcing Environment Name and CIDR Format with Variable Validation</summary>

**Situation:**  
A platform team maintains a shared Terraform module for VPC provisioning. Engineers pass in `environment` and `cidr_block` as variables. Without validation, a developer might pass `"production"` instead of the expected `"prod"`, or an invalid CIDR like `"10.0.0.0"` (missing prefix length), causing downstream failures that are hard to trace.

**HCL Configuration:**

```hcl
variable "environment" {
  type        = string
  description = "Deployment environment"

  validation {
    condition     = contains(["dev", "staging", "prod"], var.environment)
    error_message = "environment must be one of: dev, staging, prod. Got: ${var.environment}"
  }
}

variable "cidr_block" {
  type        = string
  description = "VPC CIDR block"

  validation {
    condition     = can(cidrhost(var.cidr_block, 0))
    error_message = "cidr_block must be a valid IPv4 CIDR notation (e.g. 10.0.0.0/16)."
  }
}

resource "aws_vpc" "main" {
  cidr_block = var.cidr_block
  tags = { Environment = var.environment }
}
```

**Expected Outcome:**  
Running `terraform plan -var environment=production` immediately fails with a clear, human-readable error before Terraform contacts AWS. The developer understands what's wrong and corrects the input.

**Exam Sub-Objective:** Variable `validation` block — `condition` with `contains()` and `can()`; early input validation before plan.

</details>

<details>
<summary>Scenario 2: AMI Architecture Precondition in a Shared Module</summary>

**Situation:**  
A shared EC2 module accepts `ami_id` and `instance_type` as inputs. An ARM-based instance type (`t4g.micro`) was accidentally paired with an x86_64 AMI, causing instance creation to fail with a cryptic AWS API error. Adding a `precondition` makes the failure immediate and self-documenting.

**HCL Configuration:**

```hcl
variable "ami_id"        { type = string }
variable "instance_type" { type = string }

data "aws_ami" "selected" {
  owners = ["self", "amazon"]
  filter {
    name   = "image-id"
    values = [var.ami_id]
  }
}

locals {
  arm_instance_families = ["t4g", "m6g", "c6g", "r6g"]
  instance_family       = split(".", var.instance_type)[0]
  is_arm_instance       = contains(local.arm_instance_families, local.instance_family)
}

resource "aws_instance" "app" {
  ami           = var.ami_id
  instance_type = var.instance_type

  lifecycle {
    precondition {
      condition = (
        (local.is_arm_instance && data.aws_ami.selected.architecture == "arm64") ||
        (!local.is_arm_instance && data.aws_ami.selected.architecture == "x86_64")
      )
      error_message = "AMI architecture (${data.aws_ami.selected.architecture}) does not match instance family (${local.instance_family}). Use a matching AMI."
    }
  }
}
```

**Expected Outcome:**  
If a developer passes `ami_id = "ami-x86_64"` with `instance_type = "t4g.micro"`, the plan errors immediately with a clear message before any AWS resource is created.

**Exam Sub-Objective:** `precondition` — checked before resource creation; hard error; self-documenting module assumptions.

</details>

<details>
<summary>Scenario 3: ALB Health Check Monitoring with a check Block</summary>

**Situation:**  
A team provisions an Application Load Balancer and a target group. They want to verify the ALB is forwarding traffic to healthy targets after each apply, but they don't want the apply to fail if the application is slow to start (deployment can take 2-3 minutes). A `check` block is the right tool — it warns without blocking.

**HCL Configuration:**

```hcl
resource "aws_lb" "api" {
  name               = "api-alb"
  load_balancer_type = "application"
  subnets            = var.public_subnet_ids
  security_groups    = [aws_security_group.alb.id]
}

resource "aws_lb_listener" "https" {
  load_balancer_arn = aws_lb.api.arn
  port              = 443
  protocol          = "HTTPS"
  certificate_arn   = aws_acm_certificate.api.arn

  default_action {
    type             = "forward"
    target_group_arn = aws_lb_target_group.api.arn
  }
}

check "api_endpoint_alive" {
  data "http" "health" {
    url = "https://${aws_lb.api.dns_name}/health"
    # insecure = true if cert not yet trusted
  }

  assert {
    condition     = data.http.health.status_code == 200
    error_message = "API ALB health check returned ${data.http.health.status_code} — application may still be starting."
  }
}
```

**Expected Outcome:**  
If the app is ready: apply completes silently. If not: apply completes with a warning, alerting the operator to check the application startup without rolling anything back.

**Exam Sub-Objective:** `check` block — soft assertion producing warning; scoped `data` block inside check; Terraform 1.5+ feature.

</details>

<details>
<summary>Scenario 4: Database Credentials from Vault — No Secrets in Version Control</summary>

**Situation:**  
A security audit finds that a team has been storing `db_password` in `terraform.tfvars` which is committed to Git. The remediation plan: move credentials to HashiCorp Vault KV and read them dynamically in Terraform. The `.tfvars` file is removed and the Vault provider is configured.

**HCL Configuration:**

```hcl
# ── providers.tf ─────────────────────────────────────────────────────────────
terraform {
  required_providers {
    aws   = { source = "hashicorp/aws",   version = "~> 5.0" }
    vault = { source = "hashicorp/vault", version = "~> 3.0" }
  }
}

provider "vault" {
  address = "https://vault.internal.example.com"
  # VAULT_TOKEN env var used for auth in CI/CD pipeline
}

# ── main.tf ──────────────────────────────────────────────────────────────────
data "vault_kv_secret_v2" "db" {
  mount = "secret"
  name  = "prod/postgres"
}

resource "aws_db_instance" "prod" {
  identifier        = "prod-postgres"
  engine            = "postgres"
  instance_class    = "db.t3.medium"
  allocated_storage = 100

  username = data.vault_kv_secret_v2.db.data["username"]
  password = data.vault_kv_secret_v2.db.data["password"]

  lifecycle {
    # Ignore password changes — Vault rotation will update it out-of-band
    ignore_changes = [password]
  }
}
```

**Expected Outcome:**  
No credentials in `.tfvars` or Git history. Vault provides fresh credentials at runtime. The CI/CD pipeline authenticates to Vault using a short-lived token or AppRole. Rotation is handled in Vault without Terraform needing to re-run.

**Exam Sub-Objective:** HashiCorp Vault integration via data source; benefits over `.tfvars` credential storage.

</details>

<details>
<summary>Scenario 5: Multi-Layer Validation — Variable, Precondition, and Postcondition Together</summary>

**Situation:**  
A production-grade module for RDS provisioning uses all three validation layers to create a "defence in depth" approach:
1. **Variable validation** rejects bad inputs before planning.
2. **Precondition** ensures production uses an approved instance class.
3. **Postcondition** verifies the endpoint was assigned after creation.

**HCL Configuration:**

```hcl
variable "environment" {
  type = string

  validation {
    condition     = contains(["dev", "staging", "prod"], var.environment)
    error_message = "environment must be dev, staging, or prod."
  }
}

variable "db_instance_class" {
  type    = string
  default = "db.t3.micro"

  validation {
    condition     = startswith(var.db_instance_class, "db.")
    error_message = "db_instance_class must start with 'db.' (e.g. db.t3.micro)."
  }
}

variable "db_password" {
  type      = string
  sensitive = true

  validation {
    condition     = length(var.db_password) >= 16
    error_message = "db_password must be at least 16 characters for production compliance."
  }
}

resource "aws_db_instance" "primary" {
  identifier        = "${var.environment}-postgres"
  engine            = "postgres"
  instance_class    = var.db_instance_class
  allocated_storage = 50
  username          = "admin"
  password          = var.db_password

  lifecycle {
    # Layer 2: before creation — prod must not use micro instances
    precondition {
      condition     = var.environment != "prod" || !endswith(var.db_instance_class, ".micro")
      error_message = "Production RDS must not use a .micro instance class. Use db.t3.small or larger."
    }

    # Layer 3: after creation — endpoint must be populated
    postcondition {
      condition     = self.endpoint != ""
      error_message = "RDS instance created without an endpoint — check VPC subnet and availability zone configuration."
    }

    prevent_destroy = true
  }
}

output "db_endpoint" {
  value = aws_db_instance.primary.endpoint
}

output "db_connection_string" {
  value     = "postgres://admin:${var.db_password}@${aws_db_instance.primary.endpoint}/app"
  sensitive = true
}
```

**Expected Outcome:**
- Passing `environment=production` → fails variable validation immediately (wrong value)
- Passing `environment=prod, db_instance_class=db.t3.micro` → fails precondition before RDS creation
- Passing `db_password=short` → fails variable validation (too short)
- Valid inputs + successful RDS creation → postcondition passes; sensitive output masked in terminal

**Exam Sub-Objective:** All three validation layers working together; `sensitive = true` on variable and output; `prevent_destroy` lifecycle.

</details>

---
## 9. Quick-Reference Cheat Sheet

### Condition Mechanisms — At a Glance

| Mechanism | Location | Timing | On Failure | Available Since |
|---|---|---|---|---|
| `variable validation` | Inside `variable` block | Before plan | Hard error | 0.13 |
| `precondition` | Inside `lifecycle` | Before resource create/update | Hard error | 1.2 |
| `postcondition` | Inside `lifecycle` | After resource create/update | Hard error | 1.2 |
| `check` | Top-level block | After all resources applied | **Warning only** | 1.5 |

### Sensitive Data Rules

| Rule | Detail |
|---|---|
| `sensitive = true` on variable | Masks value in plan/apply output |
| `sensitive = true` on output | Masks in `terraform output`; required if output uses sensitive var |
| State file | Sensitive values are **always plaintext** in state — encrypt your backend |
| `nonsensitive()` | Removes mask — use only when value is truly not secret |
| `sensitive()` | Adds mask to any value |
| Propagation | Derived values automatically inherit sensitivity |

### Most Tested Patterns

```
⭐ variable validation + can()      — safe CIDR/type validation
⭐ check block = WARNING only       — never fails apply
⭐ precondition/postcondition       — hard errors; postcondition uses self.*
⭐ sensitive = true on output       — required when referencing sensitive var
⭐ state file = plaintext           — sensitive = true does NOT encrypt state
⭐ Vault data source                — secrets fetched at runtime, not in VCS
⭐ nonsensitive()                   — removes mask; use with caution
```

### `can()` in Validation — Quick Pattern

```hcl
# Safely validate CIDR:         can(cidrhost(var.cidr, 0))
# Safely validate numeric str:  can(tonumber(var.port))
# Safely validate regex match:  can(regex("^pattern$", var.name))
```

### `postcondition` Self Reference

```hcl
postcondition {
  condition     = self.endpoint != ""
  error_message = "..."
}
# self.* = the resource's own attributes AFTER creation
# precondition cannot use self (resource doesn't exist yet)
```